This tutorial covers the workflow of a sequence classification project with PyTorch. We'll cover the basics of sequence classification using a simple, but effective, neural bag-of-words model, and how to use the datasets/torchtext libaries to simplify data loading/preprocessing.

Import Modules

In [2]:
import collections

import datasets
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchtext
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
import tqdm

In [3]:
seed = 1234
np.random.seed(seed)
torch.manual_seed(seed)

Loading the Dataset

In [4]:
train_data, test_data = datasets.load_dataset('imdb', split=['train', 'test'])

In [5]:
train_data, test_data

(Dataset({
     features: ['text', 'label'],
     num_rows: 25000
 }),
 Dataset({
     features: ['text', 'label'],
     num_rows: 25000
 }))

In [6]:
train_data.features

{'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}

In [7]:
test_data.features

{'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}

In [8]:
train_data[0]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

Tokenization

In [9]:
# tokenizer = torchtext.data.utils.get_tokenizer('basic_english')  # from torchtext.data.utils import get_tokenizer
tokenizer = get_tokenizer('basic_english')

In [10]:
tokenizer("Hello World! How are you doing today? i'm doing fantastic!")

['hello',
 'world',
 '!',
 'how',
 'are',
 'you',
 'doing',
 'today',
 '?',
 'i',
 "'",
 'm',
 'doing',
 'fantastic',
 '!']

In [11]:
def tokenize_example(example, tokenizer, maxlength):
    tokens = tokenizer(example['text'])[:maxlength]
    return {'tokens': tokens}

In [12]:
tokenize_example(train_data[0], tokenizer, maxlength=200)

{'tokens': ['i',
  'rented',
  'i',
  'am',
  'curious-yellow',
  'from',
  'my',
  'video',
  'store',
  'because',
  'of',
  'all',
  'the',
  'controversy',
  'that',
  'surrounded',
  'it',
  'when',
  'it',
  'was',
  'first',
  'released',
  'in',
  '1967',
  '.',
  'i',
  'also',
  'heard',
  'that',
  'at',
  'first',
  'it',
  'was',
  'seized',
  'by',
  'u',
  '.',
  's',
  '.',
  'customs',
  'if',
  'it',
  'ever',
  'tried',
  'to',
  'enter',
  'this',
  'country',
  ',',
  'therefore',
  'being',
  'a',
  'fan',
  'of',
  'films',
  'considered',
  'controversial',
  'i',
  'really',
  'had',
  'to',
  'see',
  'this',
  'for',
  'myself',
  '.',
  'the',
  'plot',
  'is',
  'centered',
  'around',
  'a',
  'young',
  'swedish',
  'drama',
  'student',
  'named',
  'lena',
  'who',
  'wants',
  'to',
  'learn',
  'everything',
  'she',
  'can',
  'about',
  'life',
  '.',
  'in',
  'particular',
  'she',
  'wants',
  'to',
  'focus',
  'her',
  'attentions',
  'to',
  '

In [13]:
maxlength = 256
train_data = train_data.map(tokenize_example, fn_kwargs={'tokenizer': tokenizer ,'maxlength': maxlength})
test_data = test_data.map(tokenize_example, fn_kwargs={'tokenizer': tokenizer ,'maxlength': maxlength})

In [14]:
train_data

Dataset({
    features: ['text', 'label', 'tokens'],
    num_rows: 25000
})

In [15]:
train_data.features

{'text': Value('string'),
 'label': ClassLabel(names=['neg', 'pos']),
 'tokens': List(Value('string'))}

In [16]:
train_data[0]['tokens'][0:25]

['i',
 'rented',
 'i',
 'am',
 'curious-yellow',
 'from',
 'my',
 'video',
 'store',
 'because',
 'of',
 'all',
 'the',
 'controversy',
 'that',
 'surrounded',
 'it',
 'when',
 'it',
 'was',
 'first',
 'released',
 'in',
 '1967',
 '.']

Creating Validation Data

In [17]:
valid_size = 0.25
train_valid_data = train_data.train_test_split(test_size=valid_size)
train_data = train_valid_data['train']
valid_data = train_valid_data['test']

In [18]:
train_data.shape, valid_data.shape, test_data.shape

((18750, 3), (6250, 3), (25000, 3))

Creating a Vocabulary

In [19]:
min_freq = 5
special_token = ['<unk>', '<pad>']

vocab = build_vocab_from_iterator(
    train_data['tokens'],
    min_freq=min_freq,
    specials=special_token
)

In [20]:
vocab.lookup_token(2)

'the'

In [21]:
len(vocab)

21635

In [22]:
vocab.get_itos()[:10]

['<unk>', '<pad>', 'the', '.', ',', 'a', 'and', 'of', 'to', "'"]

In [23]:
vocab['and']

6

In [24]:
unk_index = vocab['<unk>']
pad_index = vocab['<pad>']

In [25]:
'some_token' in vocab

False

In [26]:
'man' in vocab

True

In [27]:
vocab.set_default_index(unk_index)

In [28]:
vocab['some_token']

0

In [31]:
vocab.lookup_indices(['hello', 'world', 'some_token', '<pad>'])

[5516, 184, 0, 1]

Numericalizing Data

In [32]:
def numerical_example(example, vocab):
    ids = vocab.lookup_indices(example['tokens'])
    return {'ids': ids}

In [33]:
train_data = train_data.map(numerical_example, fn_kwargs={'vocab': vocab})
test_data = test_data.map(numerical_example, fn_kwargs={'vocab': vocab})
valid_data = valid_data.map(numerical_example, fn_kwargs={'vocab': vocab})

Map:   0%|          | 0/18750 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/6250 [00:00<?, ? examples/s]

In [34]:
train_data[0]['tokens'][0:10]

['look', ',', 'this', 'is', 'quite', 'possibly', 'one', 'of', 'the', 'best']

In [35]:
vocab.lookup_indices(train_data[0]['tokens'][0:10])

[180, 4, 14, 10, 191, 841, 34, 7, 2, 121]

In [36]:
train_data[0]['ids'][0: 10]

[180, 4, 14, 10, 191, 841, 34, 7, 2, 121]

In [37]:
train_data = train_data.with_format(type='torch', columns=['ids', 'label'])
valid_data = valid_data.with_format(type='torch', columns=['ids', 'label'])
test_data = test_data.with_format(type='torch', columns=['ids', 'label'])

In [39]:
train_data[0]['label']

tensor(1)

In [47]:
vocab.lookup_tokens(train_data[0]["ids"][:10].tolist())

['look', ',', 'this', 'is', 'quite', 'possibly', 'one', 'of', 'the', 'best']

Creating Data Loaders

In [48]:
def get_collate_fn(pad_index):
    def collate_fn(batch):
        batch_ids = [i['ids'] for i in batch]
        batch_ids = nn.utils.rnn.pad_sequence(batch_ids,padding_value=pad_index, batch_first=True)
        batch_labels = [i['label'] for i in batch]
        batch_labels = torch.stack(batch_labels)
        return {'ids': batch_ids, 'labels': batch_labels}

    return collate_fn

In [51]:
def get_data_loader(dataset, batch_size, pad_index, shuffle=False):
    collate_fn = get_collate_fn(pad_index)
    data_loader = torch.utils.data.DataLoader(dataset=dataset,
                                              batch_size=batch_size,
                                              collate_fn=collate_fn,
                                              shuffle=shuffle)
    return data_loader

In [52]:
batch_size = 512

train_data_loader = get_data_loader(train_data, batch_size, pad_index, shuffle=True)
valid_data_loader = get_data_loader(valid_data, batch_size, pad_index)
test_data_loader = get_data_loader(test_data, batch_size, pad_index)

Building the Model

In [54]:
class NBoW(nn.Module):
    def __init__(self, vocab_size, embedding_dim, output_dim, pad_index):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_index)
        self.fc = nn.Linear(embedding_dim, output_dim)

    def forward(self, ids):
        embedded = self.embedding(ids)
        pooled = embedded.mean(dim=1)
        prediction = self.fc(pooled)
        return prediction

In [55]:
vocab_size = len(vocab)
output_dim = len(train_data.unique('label'))
embedding_dim = 300

model = NBoW(vocab_size, embedding_dim, output_dim, pad_index)

In [64]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'The model has {count_parameters(model):,} trainable parameters')

The model has 6,491,102 trainable parameters


In [65]:
vectors = torchtext.vocab.GloVe()

.vector_cache\glove.840B.300d.zip:   1%|          | 26.2M/2.18G [00:16<22:21, 1.60MB/s]    


KeyboardInterrupt: 